# Amazon EMR Serverless 7.13 + Spark Connect — getting started

Write PySpark in a local Jupyter notebook and run it on Amazon EMR Serverless. The notebook is the client; EMR Serverless provisions the Spark driver and executors, runs the DataFrame and SQL operations, and returns the results. No cluster, no notebook server, no SSH.

This capability ships in Amazon EMR release `emr-7.13.0` (and later) via Apache Spark Connect. The notebook walks through it in six steps:

| # | Step |
|---|---|
| 1 | Install a local PySpark client that matches EMR 7.13.0's Spark version |
| 2 | Configure application ID, region, runtime role, and data location |
| 3 | Start a Spark Connect session and wait for it to be ready |
| 4 | Point a local `SparkSession` at the remote driver |
| 5 | Read parquet from S3 and run a couple of aggregations |
| 6 | Open the Spark UI, then terminate the session |

**Prerequisites**

- AWS credentials with `emr-serverless:StartSession`, `GetSession`, `GetSessionEndpoint`, `TerminateSession`, `GetResourceDashboard`, and `iam:PassRole` on the runtime role.
- An EMR Serverless application on `emr-7.13.0` or later, with `interactiveConfiguration.sessionEnabled = true`.
- A runtime role that the application can assume to read your S3 data.

**Documentation:** https://docs.aws.amazon.com/emr/latest/EMR-Serverless-UserGuide/spark-connect.html


## 1. Install the matching PySpark client

The local PySpark version must match the Spark version on EMR Serverless. For `emr-7.13.0`, that is Spark `3.5.6`.

In [1]:
%pip install -q "pyspark[connect]==3.5.6" boto3


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
error: uninstall-no-record-file

× Cannot uninstall botocore None
╰─> The package's contents are unknown: no RECORD file was found for botocore.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps botocore==1.38.45


Note: you may need to restart the kernel to use updated packages.


## 2. Configure

Set the four constants below for your own account. Everything after this cell is generic.

To create a Spark Connect–enabled application, see the command in the README or the AWS documentation linked above.

In [ ]:
APPLICATION_ID = "<YOUR_APPLICATION_ID>"                              # EMR Serverless app with emr-7.13.0 and sessionEnabled=true
EXECUTION_ROLE = "arn:aws:iam::<ACCOUNT_ID>:role/<YOUR_EMRS_ROLE>"   # trust policy allows emr-serverless.amazonaws.com; has S3 + Glue read
REGION         = "us-east-1"
DATA_S3_URI    = "s3://<YOUR_BUCKET>/path/to/parquet/"               # sample parquet to read in cell 5
AWS_PROFILE    = None                                                # or a named boto3 profile

import boto3, time
sess = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION) if AWS_PROFILE else boto3.Session(region_name=REGION)
emrs = sess.client("emr-serverless")

app = emrs.get_application(applicationId=APPLICATION_ID)["application"]
print(f"Application     : {app['name']}  ({APPLICATION_ID})")
print(f"Release label   : {app['releaseLabel']}")
print(f"State           : {app['state']}")
print(f"Spark Connect   : {'ENABLED' if app['interactiveConfiguration']['sessionEnabled'] else 'DISABLED'}")


## 3. Start a Spark Connect session

`StartSession` creates a Spark driver and executors for this session on the application. With pre-initialized capacity (1 driver + 2 executors), the session is typically ready in under 90 seconds.

In [3]:
# Auto-start the app if it's stopped
if app["state"] == "STOPPED":
    emrs.start_application(applicationId=APPLICATION_ID)
    print("Starting application...")
    while emrs.get_application(applicationId=APPLICATION_ID)["application"]["state"] != "STARTED":
        time.sleep(5)

resp = emrs.start_session(
    applicationId=APPLICATION_ID,
    executionRoleArn=EXECUTION_ROLE,
    name="spark-connect-demo",
)
SESSION_ID = resp["sessionId"]
print(f"Session ID: {SESSION_ID}")
print("Waiting for session to be ready...")

t0 = time.time()
while True:
    state = emrs.get_session(applicationId=APPLICATION_ID, sessionId=SESSION_ID)["session"]["state"]
    print(f"  [{int(time.time()-t0):3d}s] {state}")
    if state in ("IDLE", "STARTED"):
        break
    if state in ("FAILED", "TERMINATED"):
        raise RuntimeError(f"Session entered {state}")
    time.sleep(10)

print(f"Session ready in {int(time.time()-t0)}s")

Session ID: <SESSION_ID>
Waiting for session to be ready...
  [  0s] STARTING


  [ 10s] STARTING


  [ 20s] STARTING


  [ 30s] STARTING


  [ 40s] STARTING


  [ 50s] STARTING


  [ 61s] IDLE
Session ready in 61s


## 4. Connect a local `SparkSession` to the remote driver

`GetSessionEndpoint` returns a host and a one-hour authentication token. The Spark Connect URL must include `:443`, `use_ssl=true`, and `x-aws-proxy-auth=<token>`; the PySpark client will not connect without them.

In [4]:
from pyspark.sql import SparkSession

ep = emrs.get_session_endpoint(applicationId=APPLICATION_ID, sessionId=SESSION_ID)
connect_url = ep["endpoint"].replace("https://", "sc://", 1) + f":443/;use_ssl=true;x-aws-proxy-auth=<REDACTED>'authToken']}"

spark = SparkSession.builder.remote(connect_url).getOrCreate()
print(f"Connected. Spark version on EMR Serverless: {spark.version}")

Connected. Spark version on EMR Serverless: 3.5.6-amzn-2


## 5. Run PySpark against S3 data

The code below executes on EMR Serverless. The driver reads the parquet from S3 and returns aggregates to this notebook. Large datasets stay on the cluster; only results cross the network.

In [5]:
from pyspark.sql.functions import col, avg, count, sum as spark_sum

trips = spark.read.parquet(DATA_S3_URI)
print(f"Total trips: {trips.count():,}")
trips.printSchema()

Total trips: 200,000


root
 |-- trip_id: long (nullable = true)
 |-- pickup_ts: long (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- pickup_zone: string (nullable = true)



In [6]:
# Trips and average fare by payment type
(trips
  .groupBy("payment_type")
  .agg(count("*").alias("trips"),
       avg("fare_amount").alias("avg_fare"),
       spark_sum("tip_amount").alias("total_tips"))
  .orderBy(col("trips").desc())
  .show(truncate=False))

+------------+-----+------------------+------------------+
|payment_type|trips|avg_fare          |total_tips        |
+------------+-----+------------------+------------------+
|credit_card |80144|61.281474096626134|1001538.5599999954|
|wallet      |39993|61.289941989848195|500451.24000000424|
|cash        |39973|61.54961023691001 |499874.1000000033 |
|voucher     |39890|61.34537152168465 |498589.9299999984 |
+------------+-----+------------------+------------------+



In [7]:
# Top pickup zones by average fare
(trips
  .groupBy("pickup_zone")
  .agg(count("*").alias("trips"),
       avg("fare_amount").alias("avg_fare"),
       avg("trip_distance_km").alias("avg_km"))
  .orderBy(col("avg_fare").desc())
  .show(truncate=False))

+-----------+-----+------------------+------------------+
|pickup_zone|trips|avg_fare          |avg_km            |
+-----------+-----+------------------+------------------+
|Midtown    |20289|61.61685839617515 |24.061012864113575|
|Upper East |20150|61.547110173697135|24.297930024813923|
|LGA        |19670|61.4873507880014  |24.203440264362083|
|Harlem     |19939|61.46070364612066 |24.105629168965276|
|Bronx      |19931|61.34478350308567 |24.11229541919627 |
|Queens     |19839|61.335195322344724|24.204327839104792|
|Financial  |20014|61.276769761167216|24.11429699210542 |
|Brooklyn   |20087|61.242990989197104|24.225885398516485|
|JFK        |19919|61.10562528239405 |24.189354385260323|
|Soho       |20162|61.0765142346991  |24.282733855768313|
+-----------+-----+------------------+------------------+



In [8]:
# Pandas conversion — small result to the local kernel
pdf = (trips
  .groupBy("payment_type")
  .agg(count("*").alias("trips"), avg("fare_amount").alias("avg_fare"))
  .orderBy(col("trips").desc())
  .toPandas())
pdf

,payment_type,trips,avg_fare
0,credit_card,80144,61.281474
1,wallet,39993,61.289942
2,cash,39973,61.549610
3,voucher,39890,61.345372


## 6. Spark UI and cleanup

`GetResourceDashboard` returns a URL to the Spark UI for this session. While the session is running, it shows the live UI (jobs, stages, executors, SQL queries). After termination, the same URL serves the Spark History Server.

In [9]:
dash = emrs.get_resource_dashboard(
    applicationId=APPLICATION_ID,
    resourceId=SESSION_ID,
    resourceType="SESSION",
)
from IPython.display import Markdown
Markdown(f"[**Open live Spark UI \u2192**]({dash['url']})")

[**Open live Spark UI →**](https://s-<SESSION_ID>.dashboard.emr-serverless.us-east-1.amazonaws.com/?authToken=<REDACTED>)

In [10]:
# Close local client, terminate remote session to stop billing
spark.stop()
emrs.terminate_session(applicationId=APPLICATION_ID, sessionId=SESSION_ID)
print(f"Session {SESSION_ID} terminated.")

Session <SESSION_ID> terminated.


## Summary

Six API calls (`StartSession`, `GetSession`, `GetSessionEndpoint`, `spark.read...`, `GetResourceDashboard`, `TerminateSession`) and one local `pip install` are all that stand between a Jupyter kernel and production-scale Spark on EMR Serverless.

To adapt to your own account, change the four constants in cell 2 and rerun. To create a Spark Connect–enabled application:

```bash
aws emr-serverless create-application \
  --type SPARK --name my-spark-connect \
  --release-label emr-7.13.0 \
  --interactive-configuration '{"sessionEnabled": true}' \
  --initial-capacity '{"Driver":{"workerCount":1,"workerConfiguration":{"cpu":"4 vCPU","memory":"16 GB"}},"Executor":{"workerCount":2,"workerConfiguration":{"cpu":"4 vCPU","memory":"16 GB"}}}'
```

Billing stops when `TerminateSession` is called. The application itself auto-stops after the configured idle timeout.